In [1]:
from pathlib import Path
import pandas as pd

print("02.18 STEP 1 - MISSING VALUE DATA INSPECTION")
print("=" * 65)

BASE_DIR = Path.cwd().parent

SALES_DIR = (
    BASE_DIR
    / "data"
    / "synthetic"
    / "cancellation_adjusted"
    / "sales_adjusted"
)

sales_files = sorted(SALES_DIR.glob("sales_*.csv"))

print("\nSales Data")
print("-" * 65)
print("Sales files found:", len(sales_files))

sales_list = []

for file in sales_files:
    df = pd.read_csv(file)
    sales_list.append(df)

sales = pd.concat(
    sales_list,
    ignore_index=True
)

print("Total sales rows:", len(sales))

print("\nColumns")
print("-" * 65)

for column in sales.columns:
    print("-", column)

print("\nCurrent Missing Values")
print("-" * 65)

missing_summary = (
    sales.isna()
    .sum()
    .sort_values(ascending=False)
)

print(missing_summary)

print("\nCurrent Missing Percentage")
print("-" * 65)

missing_pct = (
    sales.isna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
)

print(missing_pct.round(4))

print("\nData Validation")
print("-" * 65)

valid_file_count = len(sales_files) == 32
valid_row_count = len(sales) == 2_118_200

print("32 sales files:", valid_file_count)
print("2,118,200 sales rows:", valid_row_count)

if valid_file_count and valid_row_count:
    print("\n02.18 Step 1 inspection: PASSED")
else:
    print("\n02.18 Step 1 inspection: FAILED")

02.18 STEP 1 - MISSING VALUE DATA INSPECTION

Sales Data
-----------------------------------------------------------------
Sales files found: 32
Total sales rows: 2118200

Columns
-----------------------------------------------------------------
- sales_line_id
- order_id
- order_date
- order_time
- customer_id
- product_id
- store_id
- salesperson_id
- promotion_id
- sales_channel
- quantity
- unit_selling_price
- discount_pct
- payment_method
- order_status

Current Missing Values
-----------------------------------------------------------------
promotion_id          1971975
salesperson_id         529575
customer_id             63501
order_date                  0
order_time                  0
order_id                    0
sales_line_id               0
store_id                    0
product_id                  0
sales_channel               0
quantity                    0
unit_selling_price          0
discount_pct                0
payment_method              0
order_status              

In [2]:
print("02.18 STEP 2 - DEFINE MISSING VALUE SCENARIO")
print("=" * 65)

missing_value_plan = {
    "customer_id": 0.005,
    "salesperson_id": 0.005,
    "promotion_id": 0.010,
    "payment_method": 0.005,
    "discount_pct": 0.005,
    "order_time": 0.005
}

protected_columns = [
    "sales_line_id",
    "order_id",
    "order_date",
    "product_id",
    "store_id",
    "sales_channel",
    "quantity",
    "unit_selling_price",
    "order_status"
]

print("\nControlled Missing-Value Plan")
print("-" * 65)

for column, rate in missing_value_plan.items():
    print(
        f"{column}: "
        f"{rate * 100:.2f}%"
    )

print("\nProtected Columns")
print("-" * 65)

for column in protected_columns:
    print("-", column)

# Validation
plan_columns_exist = all(
    column in sales.columns
    for column in missing_value_plan
)

protected_columns_exist = all(
    column in sales.columns
    for column in protected_columns
)

rates_valid = all(
    0 < rate <= 0.05
    for rate in missing_value_plan.values()
)

no_overlap = not (
    set(missing_value_plan.keys())
    & set(protected_columns)
)

print("\nValidation")
print("-" * 65)
print("Plan columns exist:", plan_columns_exist)
print("Protected columns exist:", protected_columns_exist)
print("Missing rates valid:", rates_valid)
print("No protected-column overlap:", no_overlap)

if (
    plan_columns_exist
    and protected_columns_exist
    and rates_valid
    and no_overlap
):
    print("\n02.18 Step 2 validation: PASSED")
else:
    print("\n02.18 Step 2 validation: FAILED")

02.18 STEP 2 - DEFINE MISSING VALUE SCENARIO

Controlled Missing-Value Plan
-----------------------------------------------------------------
customer_id: 0.50%
salesperson_id: 0.50%
promotion_id: 1.00%
payment_method: 0.50%
discount_pct: 0.50%
order_time: 0.50%

Protected Columns
-----------------------------------------------------------------
- sales_line_id
- order_id
- order_date
- product_id
- store_id
- sales_channel
- quantity
- unit_selling_price
- order_status

Validation
-----------------------------------------------------------------
Plan columns exist: True
Protected columns exist: True
Missing rates valid: True
No protected-column overlap: True

02.18 Step 2 validation: PASSED


In [8]:
from pathlib import Path
import pandas as pd
import numpy as np

print("02.18 STEP 3 - INJECT CONTROLLED MISSING VALUES")
print("=" * 65)

BASE_DIR = Path.cwd().parent

SOURCE_DIR = (
    BASE_DIR
    / "data"
    / "synthetic"
)

OUTPUT_DIR = (
    BASE_DIR
    / "data"
    / "synthetic"
    / "missing_value_adjusted"
    / "sales_adjusted"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

target_rates = {
    "customer_id": 0.005,
    "salesperson_id": 0.005,
    "promotion_id": 0.010,
    "payment_method": 0.005,
    "discount_pct": 0.005,
    "order_time": 0.005
}

sales_files = sorted(SOURCE_DIR.glob("sales_*.csv"))

print("\nFiles found:", len(sales_files))

# ---------------------------------------------------------
# STEP 1 - Read file information
# ---------------------------------------------------------

file_info = []

total_rows = 0

for file in sales_files:

    df = pd.read_csv(file)

    row_count = len(df)

    total_rows += row_count

    eligible_counts = {}

    for column in target_rates:
        eligible_counts[column] = df[column].notna().sum()

    file_info.append({
        "file": file,
        "rows": row_count,
        **eligible_counts
    })


print("Total rows:", total_rows)

# ---------------------------------------------------------
# STEP 2 - Calculate GLOBAL injection targets
# ---------------------------------------------------------

global_targets = {}

for column, rate in target_rates.items():

    global_targets[column] = round(
        total_rows * rate
    )

print("\nGlobal Injection Targets")
print("-" * 65)

for column in global_targets:
    print(
        column,
        ":",
        f"{global_targets[column]:,}",
        f"({target_rates[column] * 100:.2f}%)"
    )

# ---------------------------------------------------------
# STEP 3 - Allocate injection across files
# ---------------------------------------------------------

rng = np.random.default_rng(42)

injected_totals = {
    column: 0
    for column in target_rates
}

# ---------------------------------------------------------
# STEP 4 - Process each file
# ---------------------------------------------------------

for info in file_info:

    file = info["file"]

    df = pd.read_csv(file)

    for column in target_rates:

        global_target = global_targets[column]

        # Total eligible rows across all files
        total_eligible = sum(
            x[column]
            for x in file_info
        )

        # Proportional allocation for this file
        file_target = round(
            global_target
            * info[column]
            / total_eligible
        )

        # Make sure we never select more than available
        file_target = min(
            file_target,
            info[column]
        )

        if file_target > 0:

            eligible_rows = df.index[
                df[column].notna()
            ].to_numpy()

            selected_rows = rng.choice(
                eligible_rows,
                size=file_target,
                replace=False
            )

            df.loc[selected_rows, column] = np.nan

            injected_totals[column] += file_target

    output_file = OUTPUT_DIR / file.name

    df.to_csv(
        output_file,
        index=False
    )

# ---------------------------------------------------------
# STEP 5 - Correct any rounding difference
# ---------------------------------------------------------

print("\nInjection Summary")
print("-" * 65)

for column in target_rates:

    rate = (
        injected_totals[column]
        / total_rows
    )

    print(
        f"{column}: "
        f"Newly injected={injected_totals[column]:,} | "
        f"Rate={rate * 100:.2f}% | "
        f"Expected={target_rates[column] * 100:.2f}%"
    )

print("\nOutput folder:")
print(OUTPUT_DIR)

print("\n02.18 Step 3 injection: COMPLETED")

02.18 STEP 3 - INJECT CONTROLLED MISSING VALUES

Files found: 32
Total rows: 2118200

Global Injection Targets
-----------------------------------------------------------------
customer_id : 10,591 (0.50%)
salesperson_id : 10,591 (0.50%)
promotion_id : 21,182 (1.00%)
payment_method : 10,591 (0.50%)
discount_pct : 10,591 (0.50%)
order_time : 10,591 (0.50%)

Injection Summary
-----------------------------------------------------------------
customer_id: Newly injected=10,592 | Rate=0.50% | Expected=0.50%
salesperson_id: Newly injected=10,589 | Rate=0.50% | Expected=0.50%
promotion_id: Newly injected=21,182 | Rate=1.00% | Expected=1.00%
payment_method: Newly injected=10,588 | Rate=0.50% | Expected=0.50%
discount_pct: Newly injected=10,588 | Rate=0.50% | Expected=0.50%
order_time: Newly injected=10,588 | Rate=0.50% | Expected=0.50%

Output folder:
c:\Users\HP\OneDrive\Desktop\retail_sales_analytics\data\synthetic\missing_value_adjusted\sales_adjusted

02.18 Step 3 injection: COMPLETED


In [9]:
from pathlib import Path
import pandas as pd

print("02.18 STEP 4 - VALIDATE MISSING VALUE SCENARIOS")
print("=" * 65)

BASE_DIR = Path.cwd().parent

ADJUSTED_DIR = (
    BASE_DIR
    / "data"
    / "synthetic"
    / "missing_value_adjusted"
    / "sales_adjusted"
)

target_columns = [
    "customer_id",
    "salesperson_id",
    "promotion_id",
    "payment_method",
    "discount_pct",
    "order_time"
]

protected_columns = [
    "sales_line_id",
    "order_id",
    "order_date",
    "product_id",
    "store_id",
    "sales_channel",
    "quantity",
    "unit_selling_price",
    "order_status"
]

sales_files = sorted(
    ADJUSTED_DIR.glob("sales_*.csv")
)

total_rows = 0

missing_by_column = {
    column: 0
    for column in target_columns
}

protected_missing = {
    column: 0
    for column in protected_columns
}

for file in sales_files:

    df = pd.read_csv(file)

    total_rows += len(df)

    for column in target_columns:
        missing_by_column[column] += df[column].isna().sum()

    for column in protected_columns:
        protected_missing[column] += df[column].isna().sum()


print("\nValidation Results")
print("-" * 65)

print("Sales files:", len(sales_files))
print("Total rows:", total_rows)

print("\nMissing values in target columns")
print("-" * 65)

for column, count in missing_by_column.items():

    rate = count / total_rows * 100

    print(
        f"{column}: "
        f"{count:,} missing "
        f"({rate:.2f}%)"
    )

print("\nMissing values in protected columns")
print("-" * 65)

for column, count in protected_missing.items():

    print(
        f"{column}: {count:,}"
    )


target_columns_have_missing = all(
    count > 0
    for count in missing_by_column.values()
)

protected_columns_clean = all(
    count == 0
    for count in protected_missing.values()
)

file_count_valid = len(sales_files) == 32

row_count_valid = total_rows == 2_118_200


validation_passed = (
    target_columns_have_missing
    and protected_columns_clean
    and file_count_valid
    and row_count_valid
)


print("\nFinal Validation")
print("-" * 65)

print(
    "Target columns contain missing values:",
    target_columns_have_missing
)

print(
    "Protected columns have no missing values:",
    protected_columns_clean
)

print(
    "32 sales files:",
    file_count_valid
)

print(
    "2,118,200 rows:",
    row_count_valid
)

if validation_passed:
    print("\n02.18 Step 4 validation: PASSED")
else:
    print("\n02.18 Step 4 validation: FAILED")

02.18 STEP 4 - VALIDATE MISSING VALUE SCENARIOS

Validation Results
-----------------------------------------------------------------
Sales files: 32
Total rows: 2118200

Missing values in target columns
-----------------------------------------------------------------
customer_id: 74,093 missing (3.50%)
salesperson_id: 540,164 missing (25.50%)
promotion_id: 1,993,157 missing (94.10%)
payment_method: 10,588 missing (0.50%)
discount_pct: 10,588 missing (0.50%)
order_time: 10,588 missing (0.50%)

Missing values in protected columns
-----------------------------------------------------------------
sales_line_id: 0
order_id: 0
order_date: 0
product_id: 0
store_id: 0
sales_channel: 0
quantity: 0
unit_selling_price: 0
order_status: 0

Final Validation
-----------------------------------------------------------------
Target columns contain missing values: True
Protected columns have no missing values: True
32 sales files: True
2,118,200 rows: True

02.18 Step 4 validation: PASSED


In [11]:
from pathlib import Path
import pandas as pd

print("02.18 STEP 5 - FINAL MISSING VALUE VALIDATION")
print("=" * 65)

BASE_DIR = Path.cwd().parent

ORIGINAL_DIR = (
    BASE_DIR
    / "data"
    / "synthetic"
)

ADJUSTED_DIR = (
    BASE_DIR
    / "data"
    / "synthetic"
    / "missing_value_adjusted"
    / "sales_adjusted"
)

target_columns = [
    "customer_id",
    "salesperson_id",
    "promotion_id",
    "payment_method",
    "discount_pct",
    "order_time"
]

expected_rates = {
    "customer_id": 0.005,
    "salesperson_id": 0.005,
    "promotion_id": 0.010,
    "payment_method": 0.005,
    "discount_pct": 0.005,
    "order_time": 0.005
}

sales_files = sorted(
    ADJUSTED_DIR.glob("sales_*.csv")
)

injected_missing = {
    column: 0
    for column in target_columns
}

original_missing = {
    column: 0
    for column in target_columns
}

adjusted_missing = {
    column: 0
    for column in target_columns
}

total_rows = 0

for adjusted_file in sales_files:

    original_file = ORIGINAL_DIR / adjusted_file.name

    original_df = pd.read_csv(original_file)
    adjusted_df = pd.read_csv(adjusted_file)

    total_rows += len(adjusted_df)

    for column in target_columns:

        original_count = original_df[column].isna().sum()
        adjusted_count = adjusted_df[column].isna().sum()

        original_missing[column] += original_count
        adjusted_missing[column] += adjusted_count

        injected_missing[column] += max(
            adjusted_count - original_count,
            0
        )


print("\nMissing Value Comparison")
print("-" * 65)

for column in target_columns:

    injected_rate = injected_missing[column] / total_rows
    expected_rate = expected_rates[column]

    print(
        f"{column}: "
        f"Original={original_missing[column]:,} | "
        f"Adjusted={adjusted_missing[column]:,} | "
        f"Newly Injected={injected_missing[column]:,} | "
        f"Injected Rate={injected_rate * 100:.2f}% | "
        f"Expected={expected_rate * 100:.2f}%"
    )


# ---------------------------------------------------------
# Final validation
# ---------------------------------------------------------

rate_tolerance = 0.001

injection_rate_valid = True

for column in target_columns:

    actual_rate = injected_missing[column] / total_rows
    expected_rate = expected_rates[column]

    if abs(actual_rate - expected_rate) > rate_tolerance:
        injection_rate_valid = False


file_count_valid = len(sales_files) == 32

row_count_valid = total_rows == 2_118_200

validation_passed = (
    file_count_valid
    and row_count_valid
    and injection_rate_valid
)


print("\nFinal Validation")
print("-" * 65)

print("32 sales files:", file_count_valid)
print("2,118,200 rows:", row_count_valid)
print(
    "Controlled injection rates within tolerance:",
    injection_rate_valid
)

if validation_passed:
    print("\n02.18 Step 5 validation: PASSED")
else:
    print("\n02.18 Step 5 validation: FAILED")

02.18 STEP 5 - FINAL MISSING VALUE VALIDATION

Missing Value Comparison
-----------------------------------------------------------------
customer_id: Original=63,501 | Adjusted=74,093 | Newly Injected=10,592 | Injected Rate=0.50% | Expected=0.50%
salesperson_id: Original=529,575 | Adjusted=540,164 | Newly Injected=10,589 | Injected Rate=0.50% | Expected=0.50%
promotion_id: Original=1,971,975 | Adjusted=1,993,157 | Newly Injected=21,182 | Injected Rate=1.00% | Expected=1.00%
payment_method: Original=0 | Adjusted=10,588 | Newly Injected=10,588 | Injected Rate=0.50% | Expected=0.50%
discount_pct: Original=0 | Adjusted=10,588 | Newly Injected=10,588 | Injected Rate=0.50% | Expected=0.50%
order_time: Original=0 | Adjusted=10,588 | Newly Injected=10,588 | Injected Rate=0.50% | Expected=0.50%

Final Validation
-----------------------------------------------------------------
32 sales files: True
2,118,200 rows: True
Controlled injection rates within tolerance: True

02.18 Step 5 validation: 